<a href="https://colab.research.google.com/github/Shawntribuce1234/ps2-starter.ipynb/blob/main/ps2_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS2 — The Survey That Wasn't Random

**ECON 3916 · Applied Data Analytics** · Topics 5-8 · 100 points · your role: Data Analyst, Growth team

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is data analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. You write the simulations, the bootstrap loop and the tests yourself. AI is authorised and required in Phase 4. If you cannot explain a line you submitted, you cannot submit it.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Simulate the Population You Cannot See (30 pts)

### Step 1.1: Build a world with a known answer (15 pts)

In [ ]:
# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
import numpy as np
import pandas as pd

# Set a seed so the results can be reproduced
rng = np.random.default_rng(42)

N_USERS = 50_000
BASE_BEFORE, BASE_AFTER = 3.432, 3.532

def simulate_population(base, n=N_USERS, rng=rng):
    """Simulate user engagement, support tickets, and satisfaction."""

    # Generate engagement minutes using a lognormal distribution
    engagement_minutes = rng.lognormal(mean=3.5, sigma=0.6, size=n)

    # Standardize engagement
    engagement_z = (
        (engagement_minutes - np.mean(engagement_minutes))
        / np.std(engagement_minutes)
    )

    # 22% probability of having a support ticket
    had_support_ticket = rng.binomial(1, 0.22, size=n)

    # Calculate latent satisfaction with random noise
    s = (
        base
        + 0.35 * engagement_z
        - 0.60 * had_support_ticket
        + rng.normal(0, 0.75, size=n)
    )

    # Round satisfaction to the nearest integer and keep it between 1 and 5
    satisfaction = np.clip(np.rint(s), 1, 5).astype(int)

    return pd.DataFrame({
        "engagement_minutes": engagement_minutes,
        "engagement_z": engagement_z,
        "had_support_ticket": had_support_ticket,
        "s": s,
        "satisfaction": satisfaction
    })

# Populations before and after the redesign
before = simulate_population(BASE_BEFORE)
after = simulate_population(BASE_AFTER)

# Record population means
before_mean = before["satisfaction"].mean()
after_mean = after["satisfaction"].mean()

print(f"Before redesign: {before_mean:.4f}")
print(f"After redesign:  {after_mean:.4f}")
print(f"True improvement: {after_mean - before_mean:.4f}")

Before redesign: 3.2901
After redesign:  3.3860
True improvement: 0.0959


*Your answer here.*

Before redesign: 3.2901

After redesign:  3.3860

True improvement: 0.0959

### Step 1.2: Impose the response mechanism (15 pts)

In [ ]:
from scipy.special import expit

def respond(df, a, b, rng=rng):
    """Figure out which users respond to the survey."""

    # Calculate how likely each person is to respond
    z = (
        a
        + b * (df["s"] - 3.3)
        + 0.30 * df["engagement_z"]
        - 0.25 * df["had_support_ticket"]
    )

    # Turn those numbers into probabilities between 0 and 1
    p_respond = expit(z)

    # Randomly decide who actually answers the survey
    responded = rng.uniform(size=len(df)) < p_respond

    return responded

In [ ]:
# See who responds before and after the redesign
before["responded"] = respond(before, a=-1.0, b=0.45)
after["responded"] = respond(after, a=-2.75, b=2.00)

# Find the percentage of people who responded
before_rate = before["responded"].mean()
after_rate = after["responded"].mean()

# Find the average satisfaction of people who answered
before_observed = before.loc[before["responded"], "satisfaction"].mean()
after_observed = after.loc[after["responded"], "satisfaction"].mean()

# Print the results so I can compare them
print(f"Before response rate: {before_rate:.2%}")
print(f"After response rate: {after_rate:.2%}")

print(f"Before observed satisfaction: {before_observed:.2f}")
print(f"After observed satisfaction: {after_observed:.2f}")

print(f"Observed improvement: {after_observed - before_observed:.2f}")

Before response rate: 27.54%
After response rate: 15.19%
Before observed satisfaction: 3.60
After observed satisfaction: 4.30
Observed improvement: 0.70


> **🛑 Verification checkpoint.** You should get an observed pre-redesign mean of
> about **3.60** (accept 3.55 to 3.65) and an observed post-redesign mean of
> about **4.31** (accept 4.25 to 4.35) — an apparent gap of roughly **0.71**
> points out of a true improvement of **0.10**. Response rates should fall from
> roughly 27% to roughly 15%. If your numbers are outside those bands, your
> response function or your `base` values are wrong. **Do not proceed until it
> matches** — everything after this depends on reproducing the PM's slide.

*Your answer here.*

This is an example of selection bias because happier users were more likely to respond to the survey, even though satisfaction appeared to increase by 0.70 points. Response rates dropped from 27.54% to 15.19% and a larger sample wouldn't fix this because the problem is who responds, not how many people respond.

## Phase 2: Quantify What You Actually Know (35 pts)

### Step 2.1: Uncertainty without a formula (20 pts)

In [ ]:
from scipy.stats import bootstrap
def bootstrap_by_hand(x, stat, B=2000, rng=rng):
    """Resample the data and calculate the statistic each time."""

    # Store the results from each bootstrap sample
    boot_stats = []

    # Repeat the process 2000 times
    for _ in range(B):

        # Randomly select values from the data, allowing repeats
        sample = rng.choice(x, size=len(x), replace=True)

        # Calculate the mean or median and save it
        boot_stats.append(stat(sample))

    return np.array(boot_stats)

# Get satisfaction scores from users who answered after the redesign
x = after.loc[after["responded"], "satisfaction"].to_numpy()

# Bootstrap the mean and median
boot_mean = bootstrap_by_hand(x, np.mean)
boot_median = bootstrap_by_hand(x, np.median)

# Find the 95% percentile confidence intervals
mean_ci = np.percentile(boot_mean, [2.5, 97.5])
median_ci = np.percentile(boot_median, [2.5, 97.5])

# Calculate the 95% percentile interval
scipy_percentile = bootstrap(
    (x,),
    np.mean,
    n_resamples=2000,
    method="percentile",
    random_state=42
)

# Calculate the 95% BCa interval
scipy_bca = bootstrap(
    (x,),
    np.mean,
    n_resamples=2000,
    method="BCa",
    random_state=42
)

print("Mean percentile CI:", mean_ci)
print("Median percentile CI:", median_ci)

print("Handwritten percentile CI:", mean_ci)
print("SciPy percentile CI:", scipy_percentile.confidence_interval)
print("SciPy BCa CI:", scipy_bca.confidence_interval)




Mean percentile CI: [4.2899796  4.31933987]
Median percentile CI: [4. 4.]
Handwritten percentile CI: [4.2899796  4.31933987]
SciPy percentile CI: ConfidenceInterval(low=np.float64(4.2885316572331185), high=np.float64(4.319468211135975))
SciPy BCa CI: ConfidenceInterval(low=np.float64(4.288580790426669), high=np.float64(4.319731472949849))


In [ ]:
# Calculate the median's percentile and BCa intervals
median_percentile = bootstrap(
    (x,),
    np.median,
    n_resamples=2000,
    method="percentile",
    random_state=42
)

median_bca = bootstrap(
    (x,),
    np.median,
    n_resamples=2000,
    method="BCa",
    random_state=42
)

print("Handwritten median CI:", median_ci)
print("SciPy median percentile CI:", median_percentile.confidence_interval)
print("SciPy median BCa CI:", median_bca.confidence_interval)

Handwritten median CI: [4. 4.]
SciPy median percentile CI: ConfidenceInterval(low=np.float64(4.0), high=np.float64(4.0))
SciPy median BCa CI: ConfidenceInterval(low=np.float64(nan), high=np.float64(nan))


/usr/local/lib/python3.13/dist-packages/scipy/stats/_resampling.py:147: RuntimeWarning: invalid value encountered in scalar divide
  a_hat = 1/6 * sum(nums) / sum(dens)**(3/2)
/usr/local/lib/python3.13/dist-packages/scipy/_lib/_util.py:352: DegenerateDataWarning: The BCa confidence interval cannot be calculated. This problem is known to occur when the distribution is degenerate or the statistic is np.min.
  return fun(*args, **kwargs)


*Your answer here.*

My handwritten and SciPy percentile intervals were very close with both hovering around 4.29 ~ 4.32. The BCa interval was also similar showing little difference after adjusting for bias and skewness. For the median both percentile intervals were 4 while BCa returned NaN because the median didn't vary enough across samples. This makes sense since satisfaction is measured on a scale of only 1 to 5.

### Step 2.2: Coverage is a claim you can check (15 pts)

In [ ]:
from scipy.stats import bootstrap
import numpy as np

# Use the true population mean, not the survey mean
true_mean = after["satisfaction"].mean()

# Store whether each interval contains the true mean
percentile_coverage = []
bca_coverage = []

# Repeat the survey 200 times
for _ in range(200):

    # Simulate who responds this time
    responded = respond(after, a=-2.75, b=2.00)

    # Get satisfaction scores from people who answered
    sample = after.loc[responded, "satisfaction"].to_numpy()

    # Calculate both confidence intervals
    for method, results in [
        ("percentile", percentile_coverage),
        ("BCa", bca_coverage)
    ]:
        result = bootstrap(
            (sample,),
            np.mean,
            confidence_level=0.95,
            n_resamples=999,
            method=method,
            random_state=42
        )

        # Check whether the interval includes the true mean
        low = result.confidence_interval.low
        high = result.confidence_interval.high
        results.append(low <= true_mean <= high)

# Calculate coverage rates
p_coverage = np.mean(percentile_coverage)
b_coverage = np.mean(bca_coverage)

# Calculate Monte Carlo standard errors
p_se = np.sqrt(p_coverage * (1 - p_coverage) / 200)
b_se = np.sqrt(b_coverage * (1 - b_coverage) / 200)

print(f"Percentile coverage: {p_coverage:.3f}, MC SE: {p_se:.3f}")
print(f"BCa coverage:        {b_coverage:.3f}, MC SE: {b_se:.3f}")

Percentile coverage: 0.000, MC SE: 0.000
BCa coverage:        0.000, MC SE: 0.000


*Your answer here.*
Both the percentile and BCa intervals had 0% coverage so neither captured the true population mean across 200 surveys. This happened because happier users were more likely to respond creating selection bias. Both Monte Carlo standard errors were 0, so there was no observed difference between the methods. A 95% confidence interval can be misleading when the sample is biased.

## Phase 3: Test Something Worth Testing (20 pts)

### Step 3.1: The test the PM wants

In [ ]:
from scipy import stats
import numpy as np

# Get satisfaction scores from people who answered
pre = before.loc[before["responded"], "satisfaction"].to_numpy()
post = after.loc[after["responded"], "satisfaction"].to_numpy()

# Test whether the two survey averages are different
t_stat, p_value = stats.ttest_ind(post, pre, equal_var=False)

# Find the difference between the two averages
mean_diff = np.mean(post) - np.mean(pre)

# Calculate Cohen's d to measure the size of the difference
pooled_sd = np.sqrt(
    ((len(post) - 1) * np.var(post, ddof=1) +
     (len(pre) - 1) * np.var(pre, ddof=1)) /
    (len(post) + len(pre) - 2)
)

cohens_d = mean_diff / pooled_sd

# Print the results
print(f"Mean difference: {mean_diff:.4f}")
print(f"Cohen's d: {cohens_d:.4f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6g}")

Mean difference: 0.7041
Cohen's d: 0.8637
t-statistic: 65.1660
p-value: 0


*Your answer here.*

The test showed a statistically significant increase in satisfaction with a mean difference of 0.7041, a Cohen's d of 0.8637 and a p-value of 0. Also the statistical significance isn't the issue because happier users were more likely to respond after the redesign which means the test shows a difference in survey responses but doesn't correlate to satisfaction actually imporving by that amount so the results are misleading because of selection bias.

### Step 3.2: Power, before the fact

In [ ]:
from statsmodels.stats.power import TTestIndPower
import numpy as np

# Get satisfaction scores from both surveys
pre = before.loc[before["responded"], "satisfaction"].to_numpy()
post = after.loc[after["responded"], "satisfaction"].to_numpy()

# Find the standard deviation from both groups
pooled_sd = np.sqrt((np.var(pre, ddof=1) + np.var(post, ddof=1)) / 2)

# Convert the true improvement into an effect size
effect_size = 0.10 / pooled_sd

power_model = TTestIndPower()

# Find how many responses we need per group for 80% power
n_required = power_model.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.80,
    alternative="two-sided"
)

# Find the smallest effect we can detect with our current sample sizes
detectable_d = power_model.solve_power(
    effect_size=None,
    nobs1=len(pre),
    ratio=len(post) / len(pre),
    alpha=0.05,
    power=0.80,
    alternative="two-sided"
)

detectable_points = detectable_d * pooled_sd

print(f"Required responses per group: {int(np.ceil(n_required)):,}")
print(f"Smallest detectable effect (Cohen's d): {detectable_d:.4f}")
print(f"Smallest detectable difference in points: {detectable_points:.4f}")

Required responses per group: 972
Smallest detectable effect (Cohen's d): 0.0400
Smallest detectable difference in points: 0.0315


In [ ]:
# Test whether more users actually reduces the bias
sample_sizes = [5_000, 50_000, 200_000]
results = []

# Use a separate generator so the results can be reproduced
sim_rng = np.random.default_rng(123)

for n in sample_sizes:

    # Create new users before and after the redesign
    pre_pop = simulate_population(BASE_BEFORE, n=n, rng=sim_rng)
    post_pop = simulate_population(BASE_AFTER, n=n, rng=sim_rng)

    # Decide who answers each survey
    pre_response = respond(pre_pop, a=-1.0, b=0.45, rng=sim_rng)
    post_response = respond(post_pop, a=-2.75, b=2.00, rng=sim_rng)

    # Find the true improvement using everyone
    true_diff = post_pop["satisfaction"].mean() - pre_pop["satisfaction"].mean()

    # Find the improvement using only survey responses
    observed_diff = (
        post_pop.loc[post_response, "satisfaction"].mean()
        - pre_pop.loc[pre_response, "satisfaction"].mean()
    )

    # Calculate how far the survey estimate is from the truth
    bias = observed_diff - true_diff

    results.append([n, true_diff, observed_diff, bias])

# Compare the results
results_df = pd.DataFrame(
    results,
    columns=["Population per period", "True difference", "Survey difference", "Bias"]
)

print(results_df.round(4).to_string(index=False))

 Population per period  True difference  Survey difference   Bias
                  5000           0.1104             0.7166 0.6062
                 50000           0.1055             0.7151 0.6096
                200000           0.0981             0.7083 0.6102


*Your answer here.*

The power analysis showed that we need 972 responses per group to detect a true improvement of 0.10 points with 80% power. With our current sample size, the smallest detectable difference is 0.0315 points. However, increasing the sample size wouldn't fix the selection bias because happier users are still more likely to respond. The simulation showed that even when the population increased from 5,000 to 200,000, the bias stayed around 0.61 points. This means a larger sample makes our estimates more precise but doesn't necessarily make them more accurate.

## Phase 4: AI Expansion — The Co-Pilot Era (15 pts)

**Status: AI usage AUTHORISED and REQUIRED.** You have built the simulation, the bootstrap and the power analysis by hand. Now use the **P.R.I.M.E. framework** to go further than you could in the time.

### Task 4.1: The weighting attempt (10 pts)

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

def weighted_satisfaction(df):
    """Estimate satisfaction after adjusting for survey nonresponse."""

    # Use engagement and support tickets to predict who responds
    X = df[["engagement_minutes", "had_support_ticket"]]
    y = df["responded"].astype(int)

    # Train a model to estimate response probabilities
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)

    # Find how likely each person is to respond
    probabilities = model.predict_proba(X)[:, 1]

    # Only keep the people who answered the survey
    respondents = df["responded"].to_numpy()

    # Give more weight to people who were less likely to respond
    weights = 1 / np.clip(probabilities[respondents], 0.01, 1)

    # Calculate satisfaction using the new weights
    scores = df.loc[respondents, "satisfaction"].to_numpy()
    return np.average(scores, weights=weights)

# Compare the original and corrected survey averages
observed_mean = after.loc[after["responded"], "satisfaction"].mean()
weighted_mean = weighted_satisfaction(after)
true_mean = after["satisfaction"].mean()

print(f"Observed survey mean: {observed_mean:.4f}")
print(f"Weighted survey mean: {weighted_mean:.4f}")
print(f"True population mean: {true_mean:.4f}")

# Store the weighted means from each bootstrap sample
boot_means = []
boot_rng = np.random.default_rng(123)

# Resample the users 200 times
for _ in range(200):

    # Randomly sample users with replacement
    indices = boot_rng.choice(len(after), size=len(after), replace=True)
    sample = after.iloc[indices].copy()

    # Recalculate the weights and save the weighted mean
    boot_means.append(weighted_satisfaction(sample))

# Find the 95% percentile confidence interval
ci = np.percentile(boot_means, [2.5, 97.5])

print(f"Weighted mean: {weighted_mean:.4f}")
print(f"95% bootstrap CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"True mean: {true_mean:.4f}")
print(f"Bias: {weighted_mean - true_mean:.4f}")

Observed survey mean: 4.3041
Weighted survey mean: 4.1432
True population mean: 3.3860
Weighted mean: 4.1432
95% bootstrap CI: [4.1236, 4.1653]
True mean: 3.3860
Bias: 0.7572


*Your answer here.*

The correction did not fully work because the weighted mean was 4.1432 compared to the true population mean of 3.3860. Although weighting reduced the observed mean from 4.3041, there was still a bias of 0.7572. The 95% confidence interval 4.1236 ~ 4.1653 also did not contain the true mean. This happened because satisfaction itself affects who responds, which means adjusting for engagement and support tickets alone cannot remove the selection bias.

The model first estimates how likely each person is to respond based on their engagement and support tickets. It then gives more weight to people who were less likely to respond so their answers count more toward the average. However, this doesn't fully fix the problem because the model doesn't account for how satisfaction itself affects responses.

I would use AI again because it helped me understand and implement the weighting process, but I would still check its results against the true values before trusting the correction.

### Task 4.2: The slide (5 pts)

*Your answer here.*
### Survey Results: Did the Redesign Actually Improve Satisfaction?

The survey showed an increase in satisfaction from 3.60 to 4.30 after the redesign. However, this doesn't mean satisfaction actually improved by 0.70 points because happier users were more likely to respond. Response rates also dropped from 27.54% to 15.19%, which means the survey heard from fewer people after the redesign.

After adjusting for engagement and support tickets, the weighted satisfaction mean was 4.1432 with a 95% confidence interval of [4.1236, 4.1653]. This was still much higher than the true population mean of 3.3860, showing that the correction didn't fully remove selection bias.

The survey supports that respondents reported higher satisfaction but doesn't prove the redesign caused that improvement. To fix this, I would randomly select users and collect satisfaction responses using the same survey process before and after the redesign, with follow-up outreach to reduce nonresponse. This would help make the results more representative instead of relying mainly on happier users who choose to respond.

### AI Transcript

Paste **both prompt versions exactly as you sent them** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:*

*Version 1 prompt:*
[Prep] Act as a survey statistician who works on non-response
adjustment for product research.

[Request] I have a simulated user population with satisfaction scores
and two covariates, engagement_minutes and had_support_ticket, plus a
non-response mechanism where the probability of responding increases
with satisfaction. Write Python that implements inverse-probability
weighting to correct the observed survey mean, using a logistic model
of response on the covariates.

[Iterate] numpy, pandas, scikit-learn or statsmodels. Reuse my
variable names. Return the weighted estimate and its bootstrap
interval.

[Mechanism Check] Explain precisely which assumption makes this
correction valid, and what happens to it when response depends on
satisfaction ITSELF rather than only on the covariates.

[Evaluate] Given my mechanism, will this correction recover the true
mean? Answer yes or no and justify it.




*Reply:*
import numpy as np
from sklearn.linear_model import LogisticRegression

def weighted_satisfaction(df):
    """Estimate satisfaction after adjusting for survey nonresponse."""

    # Use engagement and support tickets to predict who responds
    X = df[["engagement_minutes", "had_support_ticket"]]
    y = df["responded"].astype(int)

    # Train a model to estimate response probabilities
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)

    # Find how likely each person is to respond
    probabilities = model.predict_proba(X)[:, 1]

    # Only keep the people who answered the survey
    respondents = df["responded"].to_numpy()

    # Give more weight to people who were less likely to respond
    weights = 1 / np.clip(probabilities[respondents], 0.01, 1)

    # Calculate satisfaction using the new weights
    scores = df.loc[respondents, "satisfaction"].to_numpy()
    return np.average(scores, weights=weights)

# Compare the original and corrected survey averages
observed_mean = after.loc[after["responded"], "satisfaction"].mean()
weighted_mean = weighted_satisfaction(after)
true_mean = after["satisfaction"].mean()

print(f"Observed survey mean: {observed_mean:.4f}")
print(f"Weighted survey mean: {weighted_mean:.4f}")
print(f"True population mean: {true_mean:.4f}")

# Store the weighted means from each bootstrap sample
boot_means = []
boot_rng = np.random.default_rng(123)

# Resample the users 200 times
for _ in range(200):

    # Randomly sample users with replacement
    indices = boot_rng.choice(len(after), size=len(after), replace=True)
    sample = after.iloc[indices].copy()

    # Recalculate the weights and save the weighted mean
    boot_means.append(weighted_satisfaction(sample))

# Find the 95% percentile confidence interval
ci = np.percentile(boot_means, [2.5, 97.5])

print(f"Weighted mean: {weighted_mean:.4f}")
print(f"95% bootstrap CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"True mean: {true_mean:.4f}")
print(f"Bias: {weighted_mean - true_mean:.4f}")





*Version 2 prompt:*

[Prep] Act as a survey statistician who works on non-response adjustment for product research.

[Request] I have a simulated population of 50,000 users before and after a redesign. My DataFrames are named `before` and `after`, and my columns include `satisfaction`, `s`, `engagement_minutes`, `engagement_z`, `had_support_ticket`, and `responded`. Write Python code that uses inverse-probability weighting to correct the observed post-redesign survey mean.

[Iterate] Use numpy, pandas, and scikit-learn. Make sure to use `engagement_z` and `had_support_ticket` to predict response probability, since those are the standardized variables used in my simulation. Return the weighted satisfaction estimate and its 95% bootstrap confidence interval.

[Mechanism Check] My response probability depends directly on latent satisfaction `s`, even after accounting for engagement and support tickets. Explain why this violates the missing-at-random assumption.

[Evaluate] Compare the observed and weighted survey means against the known true post-redesign population mean of approximately 3.40. Explain whether weighting actually removes selection bias and why.

*Reply:*

import numpy as np
from sklearn.linear_model import LogisticRegression

def weighted_satisfaction(df):
    """Estimate satisfaction using inverse-probability weighting."""

    # Use engagement and support tickets to predict who responds
    X = df[["engagement_z", "had_support_ticket"]]
    y = df["responded"].astype(int)

    # Train a model to estimate response probabilities
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)

    # Find how likely each person is to respond
    probabilities = model.predict_proba(X)[:, 1]

    # Only keep the people who answered the survey
    respondents = df["responded"].to_numpy(dtype=bool)

    # Give more weight to people who were less likely to respond
    weights = 1 / np.clip(probabilities[respondents], 0.01, 1)

    # Calculate the weighted satisfaction average
    scores = df.loc[respondents, "satisfaction"].to_numpy()
    return np.average(scores, weights=weights)


# Compare the original and weighted survey averages
observed_mean = after.loc[after["responded"], "satisfaction"].mean()
weighted_mean = weighted_satisfaction(after)
true_mean = after["satisfaction"].mean()

print(f"Observed survey mean: {observed_mean:.4f}")
print(f"Weighted survey mean: {weighted_mean:.4f}")
print(f"True population mean: {true_mean:.4f}")
print(f"Bias: {weighted_mean - true_mean:.4f}")

# Store the weighted averages from each bootstrap sample
boot_means = []
boot_rng = np.random.default_rng(123)

# Repeat the bootstrap 200 times
for _ in range(200):

    # Randomly resample users with replacement
    indices = boot_rng.choice(len(after), size=len(after), replace=True)
    sample = after.iloc[indices].copy()

    # Recalculate the weighted average for each sample
    boot_means.append(weighted_satisfaction(sample))

# Find the 95% bootstrap confidence interval
ci = np.percentile(boot_means, [2.5, 97.5])

print(f"Weighted mean: {weighted_mean:.4f}")
print(f"95% bootstrap CI: [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"True population mean: {true_mean:.4f}")
print(f"Bias: {weighted_mean - true_mean:.4f}")

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; both prompt versions are in it.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 2/Econ_3916_PS2.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.